In [ ]:
!pip install transformers==4.43.3 bitsandbytes -q

In [ ]:
import sys
import gc
import glob
import inspect
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print(f"GPU 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU 이름: {torch.cuda.get_device_name(0)}")
    print(f"VRAM 전체: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


In [ ]:
# EXAONE 캐시 파일 패치 (모델 로드 전 반드시 실행)

# 1. configuration_exaone.py 패치
for f in glob.glob("/workspace/.cache/huggingface/modules/**/configuration_exaone.py", recursive=True):
    with open(f) as fp:
        code = fp.read()
    code = code.replace(
        "from transformers.modeling_rope_utils import RopeParameters\n", ""
    ).replace(
        "rope_parameters: RopeParameters | None = None,", "rope_parameters=None,"
    )
    with open(f, "w") as fp:
        fp.write(code)
print("configuration 패치 완료")

# 2. modeling_exaone.py 패치
for f in glob.glob("/workspace/.cache/huggingface/modules/**/modeling_exaone.py", recursive=True):
    with open(f) as fp:
        code = fp.read()
    code = code.replace(
        "from transformers.integrations import use_kernel_forward_from_hub, use_kernel_func_from_hub, use_kernelized_func\n", ""
    ).replace(
        "from transformers.masking_utils import create_causal_mask\n", ""
    ).replace(
        "from transformers.modeling_layers import GradientCheckpointingLayer\n", ""
    ).replace(
        "from transformers.processing_utils import Unpack\n", ""
    ).replace(
        "from transformers.utils import TransformersKwargs, auto_docstring, can_return_tuple\n",
        "from transformers.utils import logging\n"
    ).replace(
        "from transformers.utils.generic import check_model_inputs, maybe_autocast\n", ""
    ).replace(
        "from transformers.modeling_utils import ALL_ATTENTION_FUNCTIONS, PreTrainedModel\n",
        "from transformers.modeling_utils import PreTrainedModel\n"
    )
    with open(f, "w") as fp:
        fp.write(code)
print("modeling 패치 완료")


In [ ]:
def patch_causal_mask_aliases():
    patched = set()

    def make_wrapper(func):
        try:
            params = inspect.signature(func).parameters
        except Exception:
            return func
        accepts_kwargs = any(p.kind == inspect.Parameter.VAR_KEYWORD for p in params.values())
        accepts_input = "input_embeds" in params
        accepts_inputs = "inputs_embeds" in params
        accepted_names = set(params)
        if accepts_kwargs and accepts_input == accepts_inputs:
            return func

        def wrapper(*args, **kwargs):
            if accepts_inputs and "input_embeds" in kwargs and "inputs_embeds" not in kwargs:
                kwargs["inputs_embeds"] = kwargs.pop("input_embeds")
            elif accepts_input and "inputs_embeds" in kwargs and "input_embeds" not in kwargs:
                kwargs["input_embeds"] = kwargs.pop("inputs_embeds")
            if not accepts_kwargs:
                kwargs = {k: v for k, v in kwargs.items() if k in accepted_names}
            return func(*args, **kwargs)
        return wrapper

    for module in list(sys.modules.values()):
        module_name = getattr(module, "__name__", "")
        if "transformers_modules" not in module_name and "modeling_exaone" not in module_name:
            continue
        for attr in ("create_causal_mask", "create_sliding_window_causal_mask"):
            func = getattr(module, attr, None)
            if func is None or id(func) in patched:
                continue
            wrapped = make_wrapper(func)
            if wrapped is not func:
                setattr(module, attr, wrapped)
                patched.add(id(wrapped))

print("patch_causal_mask_aliases 정의 완료")


In [ ]:
previous_meeting = """
[회의록] 2024년 11월 18일 (월) 스프린트 회고 회의
참석자: 김대표, 이CTO, 박PM, 최백엔드, 정프론트, 한디자이너

논의 내용
1. 스프린트 #12 회고
   - 회원가입/로그인 기능 완료, QA 통과
   - 대시보드 UI 3일 지연 (디자인 시안 수정 반복)
   - API 응답속도 800ms 초과, 최적화 필요
2. 투자 업데이트
   - 시리즈A 투자사 2곳 미팅 완료
   - MAU 지표와 리텐션 데이터 추가 요청
3. 팀 운영
   - 백엔드 개발자 1명 채용 결정
   - 온보딩 프로세스 문서화 미비

액션 아이템
- [이CTO] API 병목 구간 분석 보고서
- [박PM] 투자사 요청 지표 항목 정리
- [한디자이너] 대시보드 UI 확정 시안 공유
"""

external_document = """
[월간 서비스 지표] 2024년 11월
- MAU: 4,200명 (+18%)
- 7일 리텐션: 34%
- 유료 전환율: 3.2%
- 3일 내 이탈율: 41% (온보딩 개선 시급)
- 모바일 크래시율: 1.8% (목표 1% 이하)
- 고객 문의 응답: 평균 14시간 (목표 6시간)
- 경쟁사 A: 유사 기능 베타 출시 예정 (12월)
"""

unresolved_tasks = """
[미해결 태스크]
1. [높음] API 응답속도 최적화 - 담당: 최백엔드 (목표: 300ms)
2. [높음] 투자사 지표 대시보드 - 담당: 박PM+정프론트 (마감: 12월 5일)
3. [중간] 온보딩 플로우 개선 - 담당: 한디자이너+정프론트
4. [중간] 백엔드 개발자 채용 - 담당: 김대표+이CTO
5. [낮음] 고객 문의 대응 개선 - 담당: 박PM
"""

meeting_info = """
회의명: 12월 첫째 주 전체 팀 스탠드업
일시: 2024년 12월 2일 (월) 오전 10시
참석자: 전체 팀원 6명 / 예상 소요시간: 1시간
"""

print("더미 데이터 로드 완료")


In [ ]:
for module_name in list(sys.modules.keys()):
    if "exaone" in module_name.lower():
        del sys.modules[module_name]
gc.collect()
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

MODEL_ID = "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct"
print(f"로드 중: {MODEL_ID}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model.eval()
patch_causal_mask_aliases()

if torch.cuda.is_available():
    used = torch.cuda.memory_allocated() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM 사용: {used:.2f} GB / {total:.1f} GB")
print("로드 완료")


In [ ]:
def generate(messages, max_new_tokens=600):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text=text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated_ids = outputs[0][inputs['input_ids'].shape[-1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)


generation_messages = [
    {
        "role": "system",
        "content": "당신은 회의 기초안건을 작성하는 전문 비서입니다. 주어진 자료를 분석하여 실용적이고 구체적인 회의 안건을 작성하세요."
    },
    {
        "role": "user",
        "content": (
            "아래 자료를 바탕으로 회의 기초안건을 작성해주세요.\n\n"
            f"[회의 정보]\n{meeting_info}\n\n"
            f"[이전 회의록]\n{previous_meeting}\n\n"
            f"[월간 서비스 지표]\n{external_document}\n\n"
            f"[미해결 태스크]\n{unresolved_tasks}\n\n"
            "위 자료를 종합하여 이번 회의에서 반드시 다뤄야 할 안건을 작성해주세요. "
            "각 안건에는 논의 목적과 주요 논의 포인트를 포함해주세요."
        )
    }
]

print("기초안건 생성 중...")
generated_agenda = generate(generation_messages)

print("\n" + "="*60)
print("[생성된 기초안건]")
print("="*60)
print(generated_agenda)


In [ ]:
evaluation_messages = [
    {
        "role": "system",
        "content": "당신은 회의 안건의 품질을 평가하는 전문가입니다. 주어진 기준에 따라 객관적으로 평가하세요."
    },
    {
        "role": "user",
        "content": (
            "아래 회의 안건을 4가지 기준으로 평가해주세요.\n"
            "각 항목은 1~5점으로 채점하고, 이유를 간단히 설명해주세요.\n\n"
            "[평가 기준]\n"
            "1. 주제 반영도: 이전 회의록, 지표, 미해결 태스크의 핵심 내용이 반영됐는가?\n"
            "2. 구체성: 각 안건이 실제로 논의 가능한 수준으로 구체적인가?\n"
            "3. 형식 적절성: 회의 유형(전체 팀 스탠드업, 1시간)에 맞는 분량과 형식인가?\n"
            "4. 수정 용이성: 사람이 받아서 쉽게 수정 및 보완할 수 있는 구조인가?\n\n"
            "[입력 자료 요약]\n"
            "- 주요 이슈: API 최적화, 투자사 지표 대시보드, 온보딩 개선, 채용\n"
            "- 긴급 마감: 투자사 지표 대시보드 (12월 5일)\n\n"
            f"[평가할 안건]\n{generated_agenda}\n\n"
            "형식:\n"
            "1. 주제 반영도: X/5 - (이유)\n"
            "2. 구체성: X/5 - (이유)\n"
            "3. 형식 적절성: X/5 - (이유)\n"
            "4. 수정 용이성: X/5 - (이유)\n"
            "총점: X/20\n"
            "종합 의견:"
        )
    }
]

print("평가 중...")
evaluation_result = generate(evaluation_messages, max_new_tokens=500)

print("\n" + "="*60)
print("[평가 결과]")
print("="*60)
print(evaluation_result)


In [ ]:
print("="*60)
print(f"모델: {MODEL_ID}")
print("="*60)
print("\n[생성된 기초안건]")
print(generated_agenda)
print("\n[평가 결과]")
print(evaluation_result)

del model, tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"모델 해제 완료 | 잔여 VRAM: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
